# Trích xuất audio từ 3 file parquet

In [1]:
import os
import pandas as pd
import soundfile as sf
from tqdm import tqdm

In [2]:
train_path = "../data/raw/train_115-00000-of-00001.parquet"
val1_path = "../data/raw/validation-00000-of-00002.parquet"
val2_path = "../data/raw/validation-00001-of-00002.parquet"

train = pd.read_parquet(train_path)
val1 = pd.read_parquet(val1_path)
val2 = pd.read_parquet(val2_path)

print(train.shape)
print(val1.shape)
print(val2.shape)

(115, 26)
(1017, 26)
(1016, 26)


In [3]:
train["original_split"] = "train"
val1["original_split"] = "validation"
val2["original_split"] = "validation"

dataset = pd.concat(
    [train, val1, val2],
    ignore_index=True
)

print(dataset.shape)

(2148, 27)


In [4]:
audio_root = "../data/audio"
os.makedirs(audio_root, exist_ok=True)

In [5]:
saved = 0

for _, row in tqdm(dataset.iterrows(), total=len(dataset)):

    audio = row["audio"]

    folder = os.path.join(
        audio_root,
        os.path.dirname(row["path"])
    )

    os.makedirs(folder, exist_ok=True)

    output_path = os.path.join(
        audio_root,
        row["path"]
    )

    if os.path.exists(output_path):
        continue

    with open(output_path, "wb") as f:
        f.write(audio["bytes"])

    saved += 1

print("Saved:", saved)

100%|██████████| 2148/2148 [00:01<00:00, 1748.04it/s]

Saved: 2148


In [ ]:
import glob

files = glob.glob("../data/audio/**/*.wav", recursive=True)

print("Total wav files:", len(files))
print(files[:5])

Total wav files: 2148
['../data/audio\\dev\\000d89d79734e13460acd4ac9f5a70ba.wav', '../data/audio\\dev\\002a907801c8d7e0af6e50a232925701.wav', '../data/audio\\dev\\006d735053d0ba7cf2a0a971be4126db.wav', '../data/audio\\dev\\006de2661c6a59cad50959cc97ab68c6.wav', '../data/audio\\dev\\0096024bfdf3cc018eb6c9ee357c62ae.wav']


In [8]:
import soundfile as sf

audio, sr = sf.read(files[0])

print(audio.shape)
print(sr)

(282240,)
48000


In [9]:
missing = []

for path in dataset["path"]:

    full = os.path.join(audio_root, path)

    if not os.path.exists(full):
        missing.append(path)

print("Missing:", len(missing))

Missing: 0


In [10]:
dataset["path"].str.split("/").str[0].value_counts()

path
dev          2033
train-115     115
Name: count, dtype: int64